# 🧠 Meningioma Modelling Notebook

Run from the **repo root** (`meningioma-atypier/`). Consumes `output/datasets/` from the cleaning notebook.

DDA + EDA on **unimputed** data. Multivariable modelling on **imputed** data.

<details>
<summary><b>Pipeline map</b> — notebook step → module</summary>

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, keep `output/` | `config/` loader + phase modules |
| 01 | Load handoff parquets | `dataset_handoff` · `missingness_resolution` |
| 02 | Reload schema | `schema_infer` + `config/03_schema_overrides.py` |
| 03 | EDA / model variant lists | `config/07_analysis.py` |
| 04 | DDA on unimputed cohort | `dda` · `run_dda` |
| 05 | EDA + diagnostic accuracy | `eda` · `diagnostic_accuracy` |
| 06 | Multivariable logistic (Rubin pool) | `inferential` |
| 07 | Validation + calculator JSON | `model_validation` · `model_calculator` |
| 08 | HTML report | `config/08_report_settings.py` · `report` |

Config steps 01–08 live in `config/` (loaded via `load("NN_name")`).

</details>


## 00. Setup

⚙️ Loads modelling modules. Does **not** wipe `output/` — reads cleaning handoff artifacts.

<details>
<summary>🔧 How it works</summary>

- 📦 `from heavy_machinery.config import load` plus `heavy_machinery.cleaning_phase.*` and `heavy_machinery.modelling_phase.*` imports.
- 📁 `OUTPUT_ROOT = Path("output")` — same tree as the cleaning notebook.

</details>


In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)

from pathlib import Path

from IPython.display import display

from heavy_machinery.config import load
from heavy_machinery.cleaning_phase.schema_infer import (
    infer_schema, print_schema_template, print_column_uniques, schema_summary, ColSpec,
)
from heavy_machinery.cleaning_phase.cleaning import format_table_for_display
from heavy_machinery.cleaning_phase.dda import run_dda
from heavy_machinery.cleaning_phase.missingness_resolution import load_unimputed_dataset
from heavy_machinery.cleaning_phase.dataset_handoff import detect_imputation_method
from heavy_machinery.modelling_phase.eda import screen_associations
from heavy_machinery.modelling_phase.diagnostic_accuracy import screen_diagnostic_accuracy
from heavy_machinery.modelling_phase.inferential import run_inferential_stage, preview_multivariable_cases

OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  # do not wipe — reads cleaning outputs

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None


## 01. Load prepared datasets

Requires `output/datasets/unimputed_df.parquet` and a modelling parquet from cleaning.

Uses `missingness_resolution.load_unimputed_dataset` and `dataset_handoff.detect_imputation_method`.


In [ ]:
IMPUTATION_METHOD = detect_imputation_method(OUTPUT_ROOT)
if IMPUTATION_METHOD == "mice":
    print("Using MICE-imputed dataset.")
else:
    print("Using simple-imputed dataset.")

df = load_unimputed_dataset(OUTPUT_ROOT)
df.head()


## 02. Reload schema

Re-run infer + overrides on the loaded cohort (`schema_infer` + `load("03_schema_overrides")` → `config/03_schema_overrides.py`).


In [ ]:
schema = infer_schema(df)
schema_summary(schema)


In [ ]:
print_schema_template(schema)


In [ ]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
#print_column_uniques(df, schema)

In [ ]:
#🟧🟧🟧 Edit overrides, then run

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id", keep=False),
    
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False, datetime_bin='year'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,), keep=False),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False, datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    
    'additional_ct': ColSpec(name='additional_ct', kind='binary', replace={0.0: False, 1.0: pd.NA, 3.0: True}),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary', keep=False),
    
    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 03. Analysis configuration

Three separate lists — EDA screening uses the wide predictor pool; multivariable models use their own per-variant predictor sets.

Resolved via `config/07_analysis.py` (`load("07_analysis")`).


In [ ]:
# 🟧🟧🟧 Copy-pasteable column names from the loaded cohort
load("07_analysis").print_copy_pasteable_columns(df)

### 🎯 EDA


In [ ]:
EDA_TARGETS = ['high_grade', 'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis']
EDA_PREDICTORS = [
    'age',
    'age_bins',
    'sex',

    #'who_grade', ==> TARGET
    #'high_grade', ==> TARGET

    #'progesterone_pos',
    #'ki67_pct',
    #'ki67_mid',
    #'ki67_group'
    #'brain_invasion',
    #'hist_necrosis',

    #'additional_ct',

    'side',
    'tumor_location',
    'meningioma_count',
    'multiple_meningiomas',
    'max_diameter_cm',
    'tumor_volume',

    'tumor_episode',
    'tumor_margin',
    'dural_tail',

    'perifocal_edema',
    'edema_volume_cm3',

    'mass_effect',
    'calcification',
    'cystic_component',
    'necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'sinus_invasion',
    'transfalcine_extension',

    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',

    'adc_value',
    ]

### 📚 Literature-based multivariable models


In [ ]:
# 📚 Literature-based multivariable models — published predictor sets.
# Each variant gets its own EPV bar, forest plot, VIF table, and interpretation.
# Format: (id, title, link, target, [predictors])
LITERATURE_MODEL_VARIANTS = [
    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "sex",
            "tumor_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "tumor_location",
            "tumor_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]


### 🧪 Experimental multivariable models


In [ ]:
# 🧪 Experimental multivariable models — your own predictor sets (independent of EDA_PREDICTORS).
# Add as many as you need. Each row is one model: (id, title, link, target, [predictors]).
# Grouping in the report follows this list, not the model id string.
EXPERIMENTAL_MODEL_VARIANTS = [
    (
        "experimental_model_1",
        "model 1 | high grade",
        "",
        "high_grade",
        [
            'cystic_component',
            'cortical_destruction',
            'dural_tail',
            'tumor_volume',
            'edema_volume_cm3',
            'hyperostosis',
            'mass_effect',
            'adc_value',
            'tumor_margin',
        ],
    ),
    (
        "experimental_model_2",
        "model 2 | high grade",
        "",
        "high_grade",
        [
            'dwi_hyperintensity',
            'sex',
            'heterogeneous_enhancement',
            'hemorrhage',
            'sinus_invasion',
            'age_bins',
            'calcification',
            't2_hyperintensity',
            't1_hypointensity',
            'transfalcine_extension',

        ],
    ),
    (
        "try_hard_model",
        "try_hard | high grade",
        "",
        "high_grade",
        [
            'sex',
            'age_bins',
            'hyperostosis',
            'adc_value',
            'cystic_component',
            'cortical_destruction'
        ],
    ),

    # Example — uncomment and edit to fit another outcome:
    # (
    #     "experimental_ki67",
    #     "meningioma_atypier experimental | Ki-67 group",
    #     "",
    #     "ki67_group",
    #     [
    #         "age_bins",
    #         "sex",
    #         "max_diameter_cm",
    #         "perifocal_edema",
    #     ],
    # ),
]


In [ ]:
_c07 = load("07_analysis")

(
    EDA_TARGETS,
    EDA_PREDICTORS,
    EDA_POSITIVE_CLASS,
) = _c07.resolve_eda(df, EDA_TARGETS, EDA_PREDICTORS)

INFERENTIAL_MODEL_VARIANTS = _c07.resolve_inferential_variants(
    df,
    LITERATURE_MODEL_VARIANTS,
    EXPERIMENTAL_MODEL_VARIANTS,
)
(
    INFERENTIAL_TARGETS,
    INFERENTIAL_POSITIVE_CLASS,
) = _c07.resolve_inferential_targets(df, INFERENTIAL_MODEL_VARIANTS)


## 04. DDA on unimputed data

`dda.run_dda` on the unimputed handoff cohort.


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))


## 05. EDA on unimputed data

`eda.screen_associations` and `diagnostic_accuracy.screen_diagnostic_accuracy`.



Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§16) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

In [ ]:
#🟧🟧🟧 Full table
#assoc

## 06. Multivariable modelling on imputed data

`inferential.run_inferential_stage` (Rubin-pooled across MICE draws).


In [ ]:
#display(preview_multivariable_cases(
#    schema,
#    targets=INFERENTIAL_TARGETS,
#    variants=INFERENTIAL_MODEL_VARIANTS,
#    positive_class=INFERENTIAL_POSITIVE_CLASS,
#    output_root=OUTPUT_ROOT,
#    ))

In [ ]:
full_inferential_table = run_inferential_stage(
    schema,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
    )
#full_inferential_table

## 07. Validation and model outputs

Inferential artifacts are written to `output/inferential/` by §06 (`inferential`). Calculator JSON via `model_calculator`.


## 08. Build report.html

`config/08_report_settings.py` + `report`.



Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §03).
- **Module** — `load("08_report_settings")` → `config/08_report_settings.py` → `report`.


In [ ]:
REPORT_TITLE = "Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Arturs Balodis, Sigita Zālīte, Roberts Tumeļkāns, Valērija Aksjonova, Elizabete Stankeviča, Andris Zaguzovs"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [ ]:
_c08 = load("08_report_settings")
_c08.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)
_c08.print_output_summary(OUTPUT_ROOT)